In [5]:
import ollama
import json
from tqdm import tqdm
from pathlib import Path
import pandas as pd

In [6]:
df = pd.read_json(

    "data/scenarios.jsonl",

    lines=True

)

df.head()

,id,scenario
0,0,Sarah bought an extra large bag of mixed green...
1,1,"Last night, Sarah prepared a meal that include..."
2,2,John bought an extra large bag of potatoes fro...
3,3,Maria bought an extra-large baguette from the ...
4,4,Maria bought an extra large bag of baby spinac...


In [3]:
scenario = df.iloc[0]["scenario"]
scenario

"Sarah bought an extra large bag of mixed greens at the grocery store, thinking she could use all of it throughout the week. However, after a few days, half of the bag had already gone bad and spoiled. She usually tries to make salads with her greens but was running low on other ingredients. Now, Sarah has a pile of wilted lettuce that's past its prime and no longer fit for consumption or composting."

In [ ]:
prompt0 = f"""

You are an AI assistant designed to help reduce household food waste.

A user is in the following situation:

{scenario}

Generate a short, realistic, and supportive intervention message.

The intervention should:

- be practical and easy

- match the user's situation

- avoid guilt or moralizing

- avoid overly complicated cooking suggestions

- encourage reducing food waste naturally

Keep the response concise and conversational.

"""

In [11]:
prompt = f"""
You are AMAI, an assistant designed to help reduce household food waste.

A user is in the following situation:

{scenario}

Write ONE short intervention message.

The message should:
- be concise
- sound natural
- be practical and realistic
- match the user's likely behavior
- avoid emotional language
- avoid excessive friendliness
- avoid guilt or moralizing
- avoid long explanations
- avoid questions unless necessary

The intervention should feel like a subtle app suggestion, not a conversation.

Keep it under 4 sentences.
"""

In [10]:
response = ollama.chat(

    model="llama3.2",

    messages=[

        {

            "role": "user",

            "content": prompt

        }

    ],

    options={

        "temperature": 0.7,
        "num_predict": 60
        

    }

)

print(response["message"]["content"])

"Hey Sarah, did you know that wilted lettuce can still be used in soups, stews, and braises? Try adding it to your next potluck or simmering it with some veggies for a nutritious stock."


In [4]:
output_file = "data/interventions.jsonl"

In [ ]:
with open(output_file, "w") as f:

    for _, row in tqdm(df.iterrows(), total=len(df)):

        scenario = row["scenario"]

        response = ollama.chat(
            model="llama3.2",
            messages=[
                {
                    "role": "user",

                    "content": prompt1()

                }

            ],

            options={

                "temperature": 0.5,

                "num_predict": 40

            }

        )

        intervention = response["message"]["content"].strip()

        record = {

            "id": row["id"],

            "scenario": scenario,

            "intervention": intervention

        }

        f.write(json.dumps(record) + "\n")

print(f"Saved interventions to {output_file}")

100%|██████████| 100/100 [52:43<00:00, 31.64s/it]  

Saved interventions to data/interventions.jsonl


In [7]:
def prompt1 (scenario):


  prompt = f"""

  You are AMAI, an assistant designed to help reduce household food waste.

  You must respond ONLY to the specific situation below.

  Scenario:

  {scenario}

  Write ONE short intervention message that directly addresses the food item and behavior in this scenario.

  Requirements:

  - mention the actual food item from the scenario

  - tailor the suggestion to this exact situation

  - be concise and practical

  - avoid generic advice

  - avoid emotional language

  - avoid excessive friendliness

  - avoid moralizing

  - keep it under 6 sentences

  Do not mention foods that are not in the scenario.

  """

  return prompt

In [7]:
with open(output_file, "w") as f:

    for _, row in tqdm(df.iterrows(), total=len(df)):

        scenario = row["scenario"]

        response = ollama.chat(
            model="llama3.2",
            messages=[
                {
                    "role": "user",
                    "content": prompt1(scenario)
                }
            ],
            options={

                "temperature": 0.7,
                "num_predict": 100,
                

            }

        )

        intervention = response["message"]["content"].strip()

        record = {

            "id": row["id"],

            "scenario": scenario,

            "intervention": intervention

        }

        f.write(json.dumps(record) + "\n")

print(f"Saved interventions to {output_file}")

100%|██████████| 100/100 [1:23:03<00:00, 49.84s/it]

Saved interventions to data/interventions.jsonl


In [3]:
output_file = "data/interventions_gemma.jsonl"

In [8]:
!ollama list

]11;?\NAME                ID              SIZE      MODIFIED    
gemma3:4b           a2af6cc3eb7f    3.3 GB    12 days ago    
gemma3:1b           8648f39daa8f    815 MB    12 days ago    
deepseek-r1:1.5b    e0979632db5a    1.1 GB    12 days ago    
qwen2.5:0.5b        a8b0c5157701    397 MB    12 days ago    


In [9]:
with open(output_file, "w") as f:

    for _, row in tqdm(df.iterrows(), total=len(df)):

        scenario = row["scenario"]

        response = ollama.chat(
            model="gemma3:4b",
            messages=[
                {
                    "role": "user",
                    "content": prompt1(scenario)
                }
            ],
            options={

                "temperature": 0.7,
                "num_predict": 100,
                

            }

        )

        intervention = response["message"]["content"].strip()

        record = {

            "id": row["id"],

            "scenario": scenario,

            "intervention": intervention

        }

        f.write(json.dumps(record) + "\n")

print(f"Saved interventions to {output_file}")

100%|██████████| 100/100 [07:03<00:00,  4.23s/it]

Saved interventions to data/interventions_gemma.jsonl
